# OffsideX — Phase 3: Live VAR Inference & Broadcast App
## `03_inference_demo.ipynb`

1. **Cell 1:** Mounts Drive and loads `YOLOv8x`, `YOLOv8n-Pose`, and `offside_pytorch_model.pt`.
2. **Cell 2–3:** Single-image Colab Canvas calibration and live VAR output plot.
3. **Cell 4:** Launches the custom **OffsideX** Broadcast UI via FastAPI + Cloudflare Tunnel.


In [ ]:
!pip install -q ultralytics

import base64
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from google.colab import drive, files, output as colab_output
from IPython.display import display, HTML, JSON
from ultralytics import YOLO

drive.mount('/content/drive')

V2_DIR = Path('/content/drive/MyDrive/Offside_Detector/v2_var')
MODEL_PATH = V2_DIR / 'offside_pytorch_model.pt'
assert MODEL_PATH.exists(), f"Missing model checkpoint: {MODEL_PATH}"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
det_model = YOLO('yolov8x.pt')
pose_model = YOLO('yolov8n-pose.pt')


class ResidualOffsideNet(nn.Module):
    def __init__(self, ctx_dim=6):
        super().__init__()
        self.physics_head = nn.Linear(1, 1)
        self.mlp = nn.Sequential(
            nn.Linear(ctx_dim + 1, 16), nn.SiLU(), nn.Dropout(0.35),
            nn.Linear(16, 8), nn.SiLU(), nn.Dropout(0.25),
            nn.Linear(8, 1),
        )

    def forward(self, phys_x, ctx_x):
        phys_logit = self.physics_head(phys_x)
        full_in = torch.cat([phys_x, ctx_x], dim=1)
        corr_logit = 0.5 * torch.tanh(self.mlp(full_in))  # Cap MLP delta to prevent overfit
        return (phys_logit + corr_logit).squeeze(-1)


ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)
pt_model = ResidualOffsideNet(ctx_dim=6).to(device)
pt_model.load_state_dict(ckpt['model_state_dict'])
pt_model.eval()

SCALER_MEAN = ckpt['scaler_mean']
SCALER_SCALE = ckpt['scaler_scale']

LEGAL_KP = [0, 1, 2, 3, 4, 5, 6, 11, 12, 13, 14, 15, 16]  # Exclude arms per FIFA Law 11
GRASS_LOWER = np.array([36, 40, 40], dtype=np.uint8)       # Preserve yellow kits
GRASS_UPPER = np.array([85, 255, 255], dtype=np.uint8)
COLOR_WEIGHTS = np.array([0.50, 1.40, 1.40], dtype=np.float32)  # De-emphasize luminance L


def get_pitch_mask(img):
    h, w = img.shape[:2]
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    raw = cv2.inRange(hsv, np.array([30, 40, 40], dtype=np.uint8), GRASS_UPPER)
    closed = cv2.morphologyEx(raw, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15)))
    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask = np.zeros((h, w), dtype=np.uint8)
    if contours:
        hull = cv2.convexHull(max(contours, key=cv2.contourArea))
        cv2.drawContours(mask, [hull], -1, 255, thickness=cv2.FILLED)
    else:
        mask[int(h * 0.2):, :] = 255
    return cv2.erode(mask, np.ones((45, 15), dtype=np.uint8), iterations=1)  # Trim ad boards


def detect_players_and_ball(img, pitch_mask, imgsz=1280):
    h, w = img.shape[:2]
    det = det_model(img, classes=[0, 32], conf=0.10, imgsz=imgsz, verbose=False)[0]
    person_boxes, ball_candidates = [], []

    if det.boxes is not None and len(det.boxes) > 0:
        for b, cls_id, conf in zip(det.boxes.xyxy.cpu().numpy(), det.boxes.cls.cpu().numpy().astype(int), det.boxes.conf.cpu().numpy()):
            if cls_id == 0:
                person_boxes.append(b)
            elif cls_id == 32 and conf >= 0.28:
                bx, by = int((b[0] + b[2]) / 2.0), int((b[1] + b[3]) / 2.0)
                bw, bh = b[2] - b[0], b[3] - b[1]
                ar = bw / max(bh, 1e-5)
                if 0 <= bx < w and 0 <= by < h and pitch_mask[by, bx] > 0 and 0.6 <= ar <= 1.6 and max(bw, bh) <= 45:
                    ball_candidates.append((conf, np.array([float(bx), float(by)], dtype=np.float32)))

    ball_pos = sorted(ball_candidates, key=lambda x: x[0], reverse=True)[0][1] if ball_candidates else None

    players = []
    for box in person_boxes:
        cx, fy = int(np.clip((box[0] + box[2]) / 2.0, 0, w - 1)), int(np.clip(box[3] - 2, 0, h - 1))
        if pitch_mask[fy, cx] == 0:  # Foot-only check rejects sidelines
            continue
        x1, y1, x2, y2 = box.astype(int)
        bw, bh = x2 - x1, y2 - y1
        cx1, cy1 = max(0, x1 - int(0.10 * bw)), max(0, y1 - int(0.10 * bh))
        cx2, cy2 = min(w, x2 + int(0.10 * bw)), min(h, y2 + int(0.10 * bh))
        crop = img[cy1:cy2, cx1:cx2]
        if crop.size == 0:
            continue
        pose_conf = 0.15
        if crop.shape[0] < 120:  # Upscale far players for Pose
            scale = 3 if crop.shape[0] < 60 else 2
            crop = cv2.resize(crop, (crop.shape[1] * scale, crop.shape[0] * scale), interpolation=cv2.INTER_LINEAR)
            pose_conf = 0.10

        pres = pose_model(crop, conf=pose_conf, verbose=False)[0]
        got_kp = pres.keypoints is not None and len(pres.keypoints) > 0 and pres.keypoints.conf is not None
        if got_kp:
            confs_all = pres.keypoints.conf.cpu().numpy()
            best_k = int(np.argmax(confs_all.mean(axis=1)))
            sx, sy = (cx2 - cx1) / crop.shape[1], (cy2 - cy1) / crop.shape[0]
            g_xy = pres.keypoints.xy.cpu().numpy()[best_k] * np.array([sx, sy]) + np.array([cx1, cy1])
            k_conf = confs_all[best_k]
            pts = [g_xy[idx] for idx in LEGAL_KP if k_conf[idx] > 0.25]
            if pts:
                best_pt = max(pts, key=lambda pt: pt[0])
                max_x, max_pt = float(best_pt[0]), best_pt
            else:
                max_x, max_pt = float(x2), np.array([float(x2), (y1 + y2) / 2.0])
            players.append({
                'box': box, 'keypoints_xy': g_xy, 'keypoints_conf': k_conf,
                'max_x': max_x, 'max_point': max_pt, 'box_height': float(bh),
                'center_x': (x1 + x2) / 2.0, 'center_y': (y1 + y2) / 2.0,
            })
        else:
            max_x = float(box[2]) - 0.10 * float(bw)
            players.append({
                'box': box, 'keypoints_xy': None, 'keypoints_conf': None,
                'max_x': max_x, 'max_point': np.array([max_x, (y1 + y2) / 2.0]),
                'box_height': float(bh), 'center_x': (x1 + x2) / 2.0, 'center_y': (y1 + y2) / 2.0,
            })
    return players, ball_pos


def rgb_to_wlab(rgb):
    bgr = np.uint8([[[int(rgb[2]), int(rgb[1]), int(rgb[0])]]])
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2Lab)[0, 0].astype(np.float32) * COLOR_WEIGHTS


def get_jersey_lab(img, p):
    h, w = img.shape[:2]
    x1, y1, x2, y2 = p['box'].astype(int)
    bh, bw = y2 - y1, x2 - x1
    done = False
    if p['keypoints_xy'] is not None and p['keypoints_conf'] is not None:
        pts = [p['keypoints_xy'][i] for i in [5, 6, 11, 12] if p['keypoints_conf'][i] > 0.25]
        if len(pts) >= 2:
            cx, cy = float(np.mean([q[0] for q in pts])), float(np.mean([q[1] for q in pts]))
            hs = int(np.clip(0.11 * bh, 6, 30))
            rx1, rx2, ry1, ry2 = max(0, int(cx - hs)), min(w, int(cx + hs)), max(0, int(cy - hs)), min(h, int(cy + hs))
            if rx2 > rx1 and ry2 > ry1:
                done = True
    if not done:
        ry1, ry2 = max(0, y1 + int(0.20 * bh)), min(h, y1 + int(0.46 * bh))
        rx1, rx2 = max(0, x1 + int(0.28 * bw)), min(w, x1 + int(0.72 * bw))
    crop = img[ry1:ry2, rx1:rx2]
    if crop.size == 0:
        return np.array([50.0, 0.0, 0.0], dtype=np.float32)
    gm = cv2.inRange(cv2.cvtColor(crop, cv2.COLOR_BGR2HSV), GRASS_LOWER, GRASS_UPPER)
    lab = cv2.cvtColor(crop, cv2.COLOR_BGR2Lab)
    ng = cv2.bitwise_not(gm)
    if gm.sum() / 255.0 / max(crop.shape[0] * crop.shape[1], 1) < 0.70 and (ng > 0).sum() >= 4:
        return np.median(lab[ng > 0], axis=0).astype(np.float32)
    return np.median(lab.reshape(-1, 3), axis=0).astype(np.float32)


def classify_and_predict_var(img, cal):
    h, w = img.shape[:2]
    mask = get_pitch_mask(img)
    players, ball_pos = detect_players_and_ball(img, mask)
    if len(players) < 2:
        raise ValueError(f"Too few pitch players detected ({len(players)}).")

    att_lab, def_lab = rgb_to_wlab(cal['att_rgb']), rgb_to_wlab(cal['def_rgb'])
    attackers, defenders, excluded = [], [], []
    for p in players:
        c = get_jersey_lab(img, p) * COLOR_WEIGHTS
        da, dd = float(np.linalg.norm(c - att_lab)), float(np.linalg.norm(c - def_lab))
        p['d_att'], p['d_def'], p['team_dist'] = da, dd, min(da, dd)
        if min(da, dd) < 44.0 and min(da, dd) / (max(da, dd) + 1e-6) < 0.80:
            (attackers if da <= dd else defenders).append(p)
        else:
            excluded.append(p)

    if not attackers and excluded:
        pick = min(excluded, key=lambda p: p['d_att'])
        attackers.append(pick)
        excluded = [x for x in excluded if x is not pick]
    if not defenders and excluded:
        pick = min(excluded, key=lambda p: p['d_def'])
        defenders.append(pick)
        excluded = [x for x in excluded if x is not pick]
    if not attackers or not defenders:
        raise ValueError("Could not separate both attacking and defending teams.")

    gk_cands = [(p['max_x'], i) for i, p in enumerate(excluded)
                if p['max_x'] / w >= 0.65 and 0.08 <= p['center_y'] / h <= 0.92
                and p['d_att'] >= 50.0 and p['d_def'] >= 50.0]  # Distinct from both kits
    goalkeeper = excluded.pop(max(gk_cands)[1]) if gk_cands else None

    if goalkeeper is None and len(attackers) >= 2 and defenders:
        mx_d = max(p['max_x'] for p in defenders)
        ra = max(attackers, key=lambda p: p['max_x'])
        if ra['max_x'] > mx_d + 0.08 * w and ra['max_x'] >= 0.62 * w and ra['d_att'] > 26.0:
            attackers = [p for p in attackers if p is not ra]  # Identity guard avoids NumPy Eq
            goalkeeper = ra

    s = float(cal['pitch_slope'])
    proj = lambda p, ry: float(p['max_x'] + s * (ry - float(p['box'][3])))

    opp = list(defenders) + ([goalkeeper] if goalkeeper is not None else [])
    ref_y = float(np.mean([p['box'][3] for p in opp]))
    opp_s = sorted(opp, key=lambda p: proj(p, ref_y), reverse=True)
    k = 1 if (goalkeeper is not None and len(opp_s) >= 2) else 0
    d1 = opp_s[k]
    d2 = opp_s[k + 1] if len(opp_s) > k + 1 else d1
    d1_fy = float(d1['box'][3])

    # FIFA Law 11: max(2nd-last defender, ball)
    ball_is_line = False
    ref_line_x, ref_line_y, ref_line_lbl = float(d1['max_x']), d1_fy, 'DEF LINE'
    cand_attackers = list(attackers)

    if ball_pos is not None:
        ball_proj_x = float(ball_pos[0] + s * (d1_fy - float(ball_pos[1])))
        if ball_proj_x > float(d1['max_x']):
            ball_is_line = True
            ref_line_x, ref_line_y, ref_line_lbl = float(ball_pos[0]), float(ball_pos[1]), 'BALL OFFSIDE LINE'
            if len(cand_attackers) >= 2:  # Exclude passer in 2v1 breakaway
                passer = min(cand_attackers, key=lambda p: np.hypot(p['center_x'] - ball_pos[0], p['box'][3] - ball_pos[1]))
                if np.hypot(passer['center_x'] - ball_pos[0], passer['box'][3] - ball_pos[1]) < 0.08 * w:
                    cand_attackers = [p for p in cand_attackers if p is not passer]

    att_s = sorted(cand_attackers, key=lambda p: proj(p, ref_line_y), reverse=True)
    a1 = att_s[0]
    a2 = att_s[1] if len(att_s) > 1 else a1

    var_diff_x = (proj(a1, ref_line_y) - ref_line_x) / w
    sec_diff_x = (proj(a2, ref_line_y) - ref_line_x) / w
    spread_x = (float(d1['max_x']) - proj(d2, d1_fy)) / w
    y_diff = (float(a1['box'][3]) - ref_line_y) / h
    scale_r = a1['box_height'] / (d1['box_height'] + 1e-6)
    att_x_n = float(a1['max_x']) / w
    ball_diff = ((float(a1['max_x']) - ball_pos[0]) / w) if ball_pos is not None else 0.0

    phys_v = np.array([[np.tanh(var_diff_x / 0.05)]], dtype=np.float32)  # Preserve geometric 0.0
    ctx_v = np.array([[np.tanh(sec_diff_x / 0.12), y_diff, np.clip(scale_r, 0.5, 2.0), spread_x, att_x_n, np.tanh(ball_diff / 0.15)]], dtype=np.float32)
    ctx_v = (ctx_v - SCALER_MEAN) / (SCALER_SCALE + 1e-8)

    with torch.no_grad():
        logit = pt_model(torch.tensor(phys_v, device=device), torch.tensor(ctx_v, device=device))
        prob_offside = float(torch.sigmoid(logit).item())

    decision = 'OFFSIDE' if prob_offside >= 0.5 else 'ONSIDE'

    vis = img.copy()
    for p in attackers:
        b = p['box'].astype(int)
        cv2.rectangle(vis, (b[0], b[1]), (b[2], b[3]), (0, 0, 255), 2)
        cv2.circle(vis, tuple(p['max_point'].astype(int)), 5, (0, 0, 255), -1)
    for p in defenders:
        b = p['box'].astype(int)
        cv2.rectangle(vis, (b[0], b[1]), (b[2], b[3]), (255, 0, 0), 2)
        cv2.circle(vis, tuple(p['max_point'].astype(int)), 5, (255, 0, 0), -1)
    if goalkeeper:
        b = goalkeeper['box'].astype(int)
        cv2.rectangle(vis, (b[0], b[1]), (b[2], b[3]), (0, 255, 0), 3)
        cv2.putText(vis, 'GK', (b[0], max(20, b[1] - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 0), 2)

    ref_col = (255, 0, 200) if ball_is_line else (255, 0, 0)
    for x0, y0, col, lbl, ly in [
        (float(a1['max_x']), float(a1['box'][3]), (0, 0, 255), 'ATT LINE', 35),
        (ref_line_x, ref_line_y, ref_col, ref_line_lbl, 65),
    ]:
        p1 = (int(round(x0 + s * (-2 * h - y0))), -2 * h)
        p2 = (int(round(x0 + s * (3 * h - y0))), 3 * h)
        ok, c1, c2 = cv2.clipLine((0, 0, w, h), p1, p2)  # Exact clip avoids edge bending
        if ok:
            cv2.line(vis, c1, c2, col, 3)
            lx = int(np.clip(c1[0] + 6, 8, max(8, w - 165)))
            ly_draw = int(np.clip(c1[1] + ly, 25, max(25, h - 75)))
            cv2.putText(vis, lbl, (lx, ly_draw), cv2.FONT_HERSHEY_SIMPLEX, 0.65, col, 2)

    if ball_pos is not None:
        cv2.circle(vis, (int(ball_pos[0]), int(ball_pos[1])), 8, (255, 0, 255), -1)

    banner_col = (20, 20, 200) if decision == 'OFFSIDE' else (20, 150, 40)
    cv2.rectangle(vis, (12, h - 62), (min(w - 12, 620), h - 12), banner_col, -1)
    conf_pct = prob_offside * 100 if decision == 'OFFSIDE' else (1 - prob_offside) * 100
    cv2.putText(vis, f"VAR DECISION: {decision} | Confidence: {conf_pct:.1f}% | margin={var_diff_x:+.3f}",
                (22, h - 28), cv2.FONT_HERSHEY_SIMPLEX, 0.72, (255, 255, 255), 2)

    return vis, {
        'decision': decision, 'prob_offside': prob_offside, 'var_diff_x': float(var_diff_x),
        'n_att': len(attackers), 'n_def': len(defenders), 'gk': goalkeeper is not None,
        'ball_is_line': ball_is_line,
    }


print(f"Loaded models on {device} | OOF Acc={ckpt['oof_accuracy']:.2f}% | F1={ckpt['oof_f1']:.2f}%")


In [ ]:
uploaded = files.upload()
demo_img_path = list(uploaded.keys())[0]
demo_raw_img = cv2.imread(demo_img_path)
assert demo_raw_img is not None, "Failed to read uploaded image."

demo_cal = {}


def _sample_rgb(img_bgr, nx, ny, half=3):
    h, w = img_bgr.shape[:2]
    px, py = int(np.clip(round(nx * w), 0, w - 1)), int(np.clip(round(ny * h), 0, h - 1))
    patch = img_bgr[max(0, py - half):min(h, py + half + 1), max(0, px - half):min(w, px + half + 1)]
    med = np.median(patch.reshape(-1, 3), axis=0)
    return (int(med[2]), int(med[1]), int(med[0]))


def _encode_b64(img_bgr, max_w=900):
    h, w = img_bgr.shape[:2]
    if w > max_w:
        img_bgr = cv2.resize(img_bgr, (max_w, int(h * max_w / w)), interpolation=cv2.INTER_AREA)
    _, buf = cv2.imencode('.jpg', img_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 88])
    return 'data:image/jpeg;base64,' + base64.b64encode(buf.tobytes()).decode('ascii')


def _cb_demo_ready(payload):
    flip = bool(payload['user_flip'])
    img = cv2.flip(demo_raw_img, 1) if flip else demo_raw_img.copy()
    h, w = img.shape[:2]
    att_rgb = _sample_rgb(img, payload['att_nx'], payload['att_ny'])
    def_rgb = _sample_rgb(img, payload['def_nx'], payload['def_ny'])
    if payload.get('zero_slope') or payload.get('l1_nx') is None:
        slope = 0.0
    else:
        dx, dy = (payload['l2_nx'] - payload['l1_nx']) * w, (payload['l2_ny'] - payload['l1_ny']) * h
        slope = float(np.clip(dx / dy, -20.0, 20.0)) if abs(dy) > 5.0 else 0.0
    demo_cal.clear()
    demo_cal.update({'flip': flip, 'att_rgb': att_rgb, 'def_rgb': def_rgb, 'pitch_slope': slope, 'img': img})
    return JSON({'ok': True, 'slope': round(slope, 3), 'att_rgb': att_rgb, 'def_rgb': def_rgb})


colab_output.register_callback('demo_cal_done', _cb_demo_ready)
b64_str = _encode_b64(demo_raw_img)

demo_html = f"""
<div style="font-family:'Segoe UI',sans-serif; background:#181a20; color:#f1f3f5; padding:14px; border-radius:10px; max-width:940px; border:1px solid #333;">
  <div id="d-step" style="background:#252a36; padding:10px; border-radius:8px; margin-bottom:10px; font-size:15px; font-weight:bold; text-align:center;">
    1. Click an <span style="color:#ff5252;">ATTACKER (Red)</span> jersey (attack facing right)
  </div>
  <div style="text-align:center;"><canvas id="d-canvas" style="max-width:100%; border:2px solid #495057; border-radius:6px; cursor:crosshair;"></canvas></div>
  <div style="display:flex; justify-content:center; gap:8px; margin-top:12px; flex-wrap:wrap;">
    <button id="d-save"  style="background:#2e7d32; color:#fff; border:none; padding:9px 18px; border-radius:6px; font-weight:bold; cursor:pointer;">Confirm Calibration</button>
    <button id="d-flip"  style="background:#1565c0; color:#fff; border:none; padding:9px 14px; border-radius:6px; font-weight:bold; cursor:pointer;">Flip Direction</button>
    <button id="d-zero"  style="background:#6a1b9a; color:#fff; border:none; padding:9px 14px; border-radius:6px; font-weight:bold; cursor:pointer;">Vertical 0 deg</button>
    <button id="d-reset" style="background:#ef6c00; color:#fff; border:none; padding:9px 14px; border-radius:6px; font-weight:bold; cursor:pointer;">Reset</button>
  </div>
</div>
<script>
(() => {{
  let userFlip = false, zeroSlope = false, pts = [];
  const canvas = document.getElementById('d-canvas'), ctx = canvas.getContext('2d'), elStep = document.getElementById('d-step');
  const img = new Image();
  img.onload = () => {{ canvas.width = img.width; canvas.height = img.height; redraw(); }};
  img.src = "{b64_str}";
  function updateMsg() {{
    if (pts.length === 0) elStep.innerHTML = '1. Click an <span style="color:#ff5252;">ATTACKER (Red)</span> jersey';
    else if (pts.length === 1) elStep.innerHTML = '2. Click a <span style="color:#448aff;">DEFENDER (Blue)</span> jersey';
    else if (pts.length === 2 && !zeroSlope) elStep.innerHTML = '3. Click <span style="color:#ffd740;">Point 1</span> on the pitch line';
    else if (pts.length === 3 && !zeroSlope) elStep.innerHTML = '4. Click <span style="color:#ffd740;">Point 2</span> along the same line';
    else elStep.innerHTML = 'Click <span style="color:#69f0ae;">Confirm Calibration</span> then run the next cell.';
  }}
  function redraw() {{
    ctx.save(); ctx.clearRect(0, 0, canvas.width, canvas.height);
    if (userFlip) {{ ctx.translate(canvas.width, 0); ctx.scale(-1, 1); }}
    ctx.drawImage(img, 0, 0, canvas.width, canvas.height); ctx.restore();
    if (pts.length >= 4 && !zeroSlope) {{
      const x1 = pts[2].nx * canvas.width, y1 = pts[2].ny * canvas.height, x2 = pts[3].nx * canvas.width, y2 = pts[3].ny * canvas.height;
      if (Math.abs(y2 - y1) > 3) {{
        const s = (x2 - x1) / (y2 - y1);
        ctx.strokeStyle = '#ffd740'; ctx.lineWidth = 3; ctx.setLineDash([8, 5]);
        ctx.beginPath(); ctx.moveTo(x1 + s * (0 - y1), 0); ctx.lineTo(x1 + s * (canvas.height - y1), canvas.height); ctx.stroke(); ctx.setLineDash([]);
      }}
    }}
    const st = [['#ff1744','ATT'],['#2979ff','DEF'],['#ffd740','L1'],['#ffd740','L2']];
    pts.forEach((p, i) => {{
      const x = p.nx * canvas.width, y = p.ny * canvas.height;
      ctx.beginPath(); ctx.arc(x, y, 7, 0, Math.PI * 2); ctx.fillStyle = st[i][0]; ctx.fill();
      ctx.lineWidth = 2; ctx.strokeStyle = '#fff'; ctx.stroke();
      ctx.fillStyle = '#fff'; ctx.font = 'bold 12px sans-serif'; ctx.fillText(st[i][1], x + 10, y + 4);
    }});
  }}
  canvas.onclick = (e) => {{
    if (pts.length >= 4 || (zeroSlope && pts.length >= 2)) return;
    const r = canvas.getBoundingClientRect();
    pts.push({{nx: (e.clientX - r.left) / r.width, ny: (e.clientY - r.top) / r.height}});
    updateMsg(); redraw();
  }};
  document.getElementById('d-flip').onclick = () => {{ userFlip = !userFlip; pts = []; zeroSlope = false; updateMsg(); redraw(); }};
  document.getElementById('d-zero').onclick = () => {{ zeroSlope = true; pts = pts.slice(0, 2); updateMsg(); redraw(); }};
  document.getElementById('d-reset').onclick = () => {{ pts = []; zeroSlope = false; updateMsg(); redraw(); }};
  document.getElementById('d-save').onclick = async () => {{
    if (pts.length < 2 || (pts.length < 4 && !zeroSlope)) {{ alert('Complete calibration clicks first.'); return; }}
    const res = await google.colab.kernel.invokeFunction('demo_cal_done', [{{
      user_flip: userFlip, zero_slope: zeroSlope,
      att_nx: pts[0].nx, att_ny: pts[0].ny, def_nx: pts[1].nx, def_ny: pts[1].ny,
      l1_nx: pts.length >= 4 ? pts[2].nx : null, l1_ny: pts.length >= 4 ? pts[2].ny : null,
      l2_nx: pts.length >= 4 ? pts[3].nx : null, l2_ny: pts.length >= 4 ? pts[3].ny : null,
    }}], {{}});
    const d = (typeof res.data['application/json'] === 'string') ? JSON.parse(res.data['application/json']) : res.data['application/json'];
    elStep.innerHTML = `Calibration saved (slope=${{d.slope}}). Run the inference cell next.`;
    elStep.style.color = '#69f0ae';
  }};
}})();
</script>
"""
display(HTML(demo_html))


In [ ]:
assert demo_cal, "Run and confirm calibration in the previous cell first."

vis_out, res = classify_and_predict_var(demo_cal['img'], demo_cal)
print(f"Decision: {res['decision']} | P(Offside)={res['prob_offside']*100:.1f}% | margin={res['var_diff_x']:+.4f} | slope={demo_cal['pitch_slope']:+.2f}")
print(f"Detected -> ATT={res['n_att']}, DEF={res['n_def']}, GK={res['gk']}")

plt.figure(figsize=(16, 9))
plt.imshow(cv2.cvtColor(vis_out, cv2.COLOR_BGR2RGB))
plt.title(f"VAR Decision: {res['decision']} (P_offside={res['prob_offside']*100:.1f}%)", fontsize=14, fontweight='bold')
plt.axis('off'); plt.tight_layout(); plt.show()


In [ ]:
!pip install -q fastapi uvicorn python-multipart

import os
import re
import socket
import subprocess
import threading
import time
import urllib.request
from typing import Optional

from fastapi import FastAPI, File, Form, UploadFile
from fastapi.responses import FileResponse, JSONResponse
from fastapi.staticfiles import StaticFiles
import uvicorn
from IPython.display import display, HTML

assert "classify_and_predict_var" in globals(), "Run Cell 1 first to load models."

WEB_APP_DIR = V2_DIR / "web_app"
INDEX_PATH = WEB_APP_DIR / "index.html"
ASSETS_DIR = WEB_APP_DIR / "assets"
assert INDEX_PATH.exists() and ASSETS_DIR.exists(), f"Upload web_app folder to {WEB_APP_DIR}"

CF_BIN = Path("/usr/local/bin/cloudflared")
if not CF_BIN.exists():
    subprocess.run(
        ["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", str(CF_BIN)],
        check=True,
    )
    os.chmod(CF_BIN, 0o755)


def _sample_rgb_norm(img_bgr, nx, ny, half=3):
    h, w = img_bgr.shape[:2]
    px, py = int(np.clip(round(nx * w), 0, w - 1)), int(np.clip(round(ny * h), 0, h - 1))
    patch = img_bgr[max(0, py - half):min(h, py + half + 1), max(0, px - half):min(w, px + half + 1)]
    med = np.median(patch.reshape(-1, 3), axis=0)
    return (int(med[2]), int(med[1]), int(med[0]))


def _encode_b64_jpeg(img_bgr, quality=92):
    _, buf = cv2.imencode(".jpg", img_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), quality])
    return "data:image/jpeg;base64," + base64.b64encode(buf.tobytes()).decode("ascii")


offside_api = FastAPI(title="OffsideX VAR API")
offside_api.mount("/assets", StaticFiles(directory=str(ASSETS_DIR)), name="assets")


@offside_api.get("/")
def serve_home():
    return FileResponse(INDEX_PATH)


@offside_api.get("/api/health")
def api_health():
    return {
        "ok": True, "device": str(device),
        "oof_accuracy": float(ckpt.get("oof_accuracy", 0.0)),
        "oof_f1": float(ckpt.get("oof_f1", 0.0)),
    }


@offside_api.post("/api/analyze")
async def api_analyze(
    image: UploadFile = File(...),
    att_nx: float = Form(...), att_ny: float = Form(...),
    def_nx: float = Form(...), def_ny: float = Form(...),
    l1_nx: Optional[float] = Form(None), l1_ny: Optional[float] = Form(None),
    l2_nx: Optional[float] = Form(None), l2_ny: Optional[float] = Form(None),
    flipped: bool = Form(False), zero_slope: bool = Form(False),
):
    raw_bytes = await image.read()
    raw_bgr = cv2.imdecode(np.frombuffer(raw_bytes, dtype=np.uint8), cv2.IMREAD_COLOR)
    if raw_bgr is None:
        return JSONResponse(status_code=400, content={"ok": False, "error": "Could not decode image file."})

    aligned_bgr = cv2.flip(raw_bgr, 1) if flipped else raw_bgr.copy()
    h, w = aligned_bgr.shape[:2]
    att_rgb = _sample_rgb_norm(aligned_bgr, att_nx, att_ny)
    def_rgb = _sample_rgb_norm(aligned_bgr, def_nx, def_ny)

    if zero_slope or l1_nx is None or l1_ny is None or l2_nx is None or l2_ny is None:
        slope = 0.0
    else:
        dx, dy = (l2_nx - l1_nx) * w, (l2_ny - l1_ny) * h
        slope = float(np.clip(dx / dy, -20.0, 20.0)) if abs(dy) > 5.0 else 0.0

    try:
        vis_bgr, res = classify_and_predict_var(aligned_bgr, {"att_rgb": att_rgb, "def_rgb": def_rgb, "pitch_slope": slope})
    except Exception as exc:
        return JSONResponse(status_code=400, content={"ok": False, "error": str(exc)})

    prob_off = float(res["prob_offside"])
    conf_pct = (prob_off * 100.0) if res["decision"] == "OFFSIDE" else ((1.0 - prob_off) * 100.0)
    return {
        "ok": True, "decision": res["decision"], "prob_offside": round(prob_off, 4),
        "confidence": round(conf_pct, 1), "var_diff_x": round(float(res["var_diff_x"]), 4),
        "pitch_slope": round(slope, 3), "n_att": res["n_att"], "n_def": res["n_def"],
        "gk": bool(res["gk"]), "ball_is_line": bool(res.get("ball_is_line", False)),
        "result_b64": _encode_b64_jpeg(vis_bgr),
    }


def _pick_free_port(preferred=7865):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        if s.connect_ex(("127.0.0.1", preferred)) != 0:
            return preferred
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


PORT = _pick_free_port(7865)
threading.Thread(
    target=lambda: uvicorn.run(offside_api, host="127.0.0.1", port=PORT, log_level="warning"),
    daemon=True,
).start()

for _ in range(30):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health", timeout=1) as r:
            if r.status == 200:
                break
    except Exception:
        time.sleep(0.25)

proc = subprocess.Popen(
    [str(CF_BIN), "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

url_shown = False
try:
    for line in proc.stdout:
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if m and not url_shown:
            public_url = m.group(0)
            url_shown = True
            print(f"OffsideX live URL: {public_url}")
            display(HTML(f'<div style="padding:14px 18px;border-radius:10px;background:#111318;border:1px solid #f36c08;"><b style="color:#fff;">OffsideX Ready: </b><a href="{public_url}" target="_blank" style="color:#ff8524;">{public_url}</a></div>'))
except KeyboardInterrupt:
    print("Stopped OffsideX server.")
finally:
    proc.terminate()
